# Feature Importance

In [ ]:
import json
import pandas as pd

from models.encoder.common import METRICS_DIR
from evaluation.encoder.helpers import (
    export_feature_importance_latex,
    plot_feature_importance_bars,
)

FEATURE_NAME_MAPPING = {
    "extract_word_length_by_char": "Word Length",
    "extract_ngram_word_length_by_char": "Word Length Trigram",
    "extract_sentence_length_by_word": "Sentence Length",
    "extract_char_ngrams": "Character N-Gram Frequency",
    "extract_punctuation": "Punctuation Frequency",
    "extract_pos_frequency": "Morph. POS Frequency",
    "extract_pos_ngrams": "Morph. POS N-Gram Frequency",
    "extract_dependency_tree_structure": "Dep. Tree Structure",
    "extract_dependency_tree_relations": "Dep. Tree N-Gram Frequency",
    "extract_casing": "Casing-Position Frequency",
    "extract_casing_bigrams": "Casing Bigram Frequency",
    "extract_ttr": "Type-Token Ratio (TTR)",
    "extract_lexical_density": "Lexical Density",
    "extract_sentence_length_by_word_avg": "Sentence Length Average",
    "extract_syllable_count_avg": "Syllable Count Average",
    "extract_3+syllable_count_ratio": "3+ Syllable Word Ratio",
    "extract_stopwords": "Stopword Ratio",
    "extract_noun_phrase_lengths": "Noun Phrase Frequency",
    "extract_supersense": "Supersense Frequency",
    "extract_entity_categories": "Named Entity Frequency",
    "extract_events": "Event Ratio",
    "extract_places": "Place Ratio",
    "extract_tense": "Tense Ratio",
    "extract_numeric_word_ratio": "Numeric Word Ratio",
    "extract_quote_ratio": "Quote Ratio",
    "extract_contractions": "Contraction Ratio",
    "extract_articles": "Article Ratio",
    "extract_word_concreteness": "Word Concreteness Frequency",
    "extract_word_concreteness_avg": "Word Concreteness Average",
    "extract_preposition_imageability": "Preposition Imageability Frequency",
    "extract_preposition_imageability_avg": "Preposition Imageability Average",
    "extract_polysemy": "Polysemy Frequency",
}


def load_metrics(pattern):
    """Load and average metrics from JSON files matching the pattern."""
    files = list((METRICS_DIR / "feature-importance").glob(pattern))
    if not files:
        return None
    all_metrics = []
    for filepath in files:
        with open(filepath, "r") as f:
            metrics = json.load(f)
            all_metrics.append(metrics)
    if not all_metrics:
        return None
    # Average the metrics
    averaged = {}
    keys = ["mse", "accuracy", "f1"]
    for key in keys:
        if key == "f1":
            # Average element-wise
            f1_lists = [m.get("f1", []) for m in all_metrics]
            if f1_lists and all(len(lst) == len(f1_lists[0]) for lst in f1_lists):
                averaged["f1"] = [sum(x) / len(f1_lists) for x in zip(*f1_lists)]
            else:
                averaged["f1"] = []
        else:
            values = [m.get(key, 0) for m in all_metrics]
            averaged[key] = sum(values) / len(values) if values else 0
    return {"averaged": averaged, "all": all_metrics}


# Load baseline (all features)
baseline_metrics = load_metrics("catboost_minilm_catboost-minilm-s*_test_*.json")
if baseline_metrics:
    baseline_averaged = baseline_metrics["averaged"]
    baseline_all = baseline_metrics["all"]
else:
    baseline_averaged = None
    baseline_all = None

# Load features only (no embeddings)
features_only_metrics = load_metrics("catboost_minilm_features-minilm-s*_test_*.json")
if features_only_metrics:
    features_only_averaged = features_only_metrics["averaged"]
    features_only_all = features_only_metrics["all"]
else:
    features_only_averaged = None
    features_only_all = None

# Load metrics for each removed feature group
removed_group_averaged = {}
removed_group_all = {}
for group in FEATURE_NAME_MAPPING.keys():
    metrics = load_metrics(f"catboost_minilm_features-{group}-s*_test_*.json")
    if metrics:
        removed_group_averaged[group] = metrics["averaged"]
        removed_group_all[group] = metrics["all"]
    else:
        removed_group_averaged[group] = None
        removed_group_all[group] = None

print(f"Loaded baseline metrics: {baseline_averaged is not None}")
print(f"Loaded features-only metrics: {features_only_averaged is not None}")
print(
    f"Loaded {sum(1 for m in removed_group_averaged.values() if m is not None)}/{len(FEATURE_NAME_MAPPING)} removed group metrics"
)

# Debug: print averaged values
if baseline_averaged:
    print(
        f"Baseline averaged MSE: {baseline_averaged['mse']:.4f}, Accuracy: {baseline_averaged['accuracy']:.4f}"
    )
if features_only_averaged:
    print(
        f"Features-only averaged MSE: {features_only_averaged['mse']:.4f}, Accuracy: {features_only_averaged['accuracy']:.4f}"
    )

In [ ]:
import math
import numpy as np

rows = []


def macro_f1(m):
    return sum(m["f1"]) / len(m["f1"]) if m else None


def delta_stats(all_metrics, baseline_metrics, value_fn):
    if not baseline_metrics or not all_metrics:
        return float("nan"), float("nan")
    deltas = [value_fn(g) - value_fn(b) for g, b in zip(all_metrics, baseline_metrics)]
    if not deltas:
        return float("nan"), float("nan")
    mean_delta = float(np.mean(deltas))
    std_delta = float(np.std(deltas, ddof=1)) if len(deltas) > 1 else 0.0
    return mean_delta, 2 * std_delta


baseline_macro_f1 = None
if baseline_averaged:
    baseline_rmse = math.sqrt(baseline_averaged["mse"])
    baseline_macro_f1 = macro_f1(baseline_averaged)
    rows.append(
        {
            "Removed Feature Group": "None – Baseline",
            "RMSE": baseline_rmse,
            "Accuracy": baseline_averaged["accuracy"],
            "Macro Avg F1": baseline_macro_f1,
            "RMSE Δ": 0.0,
            "RMSE 2σ": 0.0,
            "Accuracy Δ": 0.0,
            "Accuracy 2σ": 0.0,
            "Macro F1 Δ": 0.0,
            "Macro F1 2σ": 0.0,
        }
    )

if features_only_averaged:
    features_only_rmse = math.sqrt(features_only_averaged["mse"])
    rmse_delta_mean, rmse_delta_2sigma = delta_stats(
        features_only_all, baseline_all, lambda m: math.sqrt(m["mse"])
    )
    acc_delta_mean, acc_delta_2sigma = delta_stats(
        features_only_all, baseline_all, lambda m: m["accuracy"]
    )
    macro_delta_mean, macro_delta_2sigma = delta_stats(
        features_only_all, baseline_all, macro_f1
    )
    rows.append(
        {
            "Removed Feature Group": "MiniLM embeddings",
            "RMSE": features_only_rmse,
            "Accuracy": features_only_averaged["accuracy"],
            "Macro Avg F1": macro_f1(features_only_averaged),
            "RMSE Δ": rmse_delta_mean,
            "RMSE 2σ": rmse_delta_2sigma,
            "Accuracy Δ": acc_delta_mean,
            "Accuracy 2σ": acc_delta_2sigma,
            "Macro F1 Δ": macro_delta_mean,
            "Macro F1 2σ": macro_delta_2sigma,
        }
    )

for group in FEATURE_NAME_MAPPING.keys():
    metrics = removed_group_averaged[group]
    all_metrics = removed_group_all[group]
    if metrics:
        human_name = FEATURE_NAME_MAPPING.get(
            group, group.replace("extract_", "").replace("_", " ").title()
        )
        rmse_delta_mean, rmse_delta_2sigma = delta_stats(
            all_metrics, baseline_all, lambda m: math.sqrt(m["mse"])
        )
        acc_delta_mean, acc_delta_2sigma = delta_stats(
            all_metrics, baseline_all, lambda m: m["accuracy"]
        )
        macro_delta_mean, macro_delta_2sigma = delta_stats(
            all_metrics, baseline_all, macro_f1
        )
        rows.append(
            {
                "Removed Feature Group": human_name,
                "RMSE": math.sqrt(metrics["mse"]),
                "Accuracy": metrics["accuracy"],
                "Macro Avg F1": macro_f1(metrics),
                "RMSE Δ": rmse_delta_mean,
                "RMSE 2σ": rmse_delta_2sigma,
                "Accuracy Δ": acc_delta_mean,
                "Accuracy 2σ": acc_delta_2sigma,
                "Macro F1 Δ": macro_delta_mean,
                "Macro F1 2σ": macro_delta_2sigma,
            }
        )

df = pd.DataFrame(rows)

# Configuration options
columns_to_include = [
    "Removed Feature Group",
    "RMSE",
    "RMSE Δ",
    "Accuracy Δ",
    "Macro F1 Δ",
]
sort_column = "RMSE Δ"
sort_ascending = False
# gradient_columns: list of columns to apply gradient. Prefix with "-" to reverse direction (higher = better)
gradient_columns = ["-RMSE Δ", "Accuracy Δ"]

# Apply selections
df = df[columns_to_include]
df = df.sort_values(sort_column, ascending=sort_ascending).reset_index(drop=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


def highlight_baselines(row):
    config = row["Removed Feature Group"]
    if "Baseline" in config:
        return ["font-weight: bold;"] * len(row)
    return [""] * len(row)


# Dynamic format dict for numeric columns
format_dict = {
    col: "{:.4f}" for col in df.columns if col not in ["Removed Feature Group"]
}

styled_df = df.style.format(format_dict).apply(highlight_baselines, axis=1)

# Apply gradients to specified columns
for grad_col in gradient_columns:
    if grad_col.startswith("-"):
        actual_col = grad_col[1:]
        cmap = "RdYlGn"  # Green to red: higher values better
    else:
        actual_col = grad_col
        cmap = "RdYlGn_r"  # Red to green: higher values worse
    if actual_col in df.columns:
        # Center the gradient at 0 (yellow) by using symmetric min/max
        abs_max = max(abs(df[actual_col].min()), abs(df[actual_col].max()))
        vmin = -abs_max
        vmax = abs_max
        styled_df = styled_df.background_gradient(
            subset=[actual_col], cmap=cmap, vmin=vmin, vmax=vmax
        )
styled_df

In [ ]:
plot_feature_importance_bars(df)

In [ ]:
export_feature_importance_latex(df)